# Lightcurve quality and detection-criteria audit

This notebook audits serialized detected sources from `ZTF_sim.py`. It separates the span of positive detections from the span of all stored observations: the former is the relevant check for the detector's 14-day qualifying window, while the latter can include non-detections outside that window.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

QUALITY = Path('results/detection_criteria_sweep/lightcurve_quality.csv')
quality = pd.read_csv(QUALITY)
quality.head()

In [ ]:
quality[['positive_span_days', 'all_observation_span_days', 'max_positive_gap_days', 'n_positive']].describe(percentiles=[.5, .9, .95, .99]).T

In [ ]:
print('Positive spans over 14 days:', int(quality['positive_span_exceeds_14_days'].sum()))
print('All-observation spans over 14 days:', int(quality['all_observation_span_exceeds_14_days'].sum()))
quality.loc[quality['positive_span_exceeds_14_days']].sort_values('positive_span_days', ascending=False).head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
quality['positive_span_days'].hist(ax=axes[0], bins=50)
axes[0].axvline(14, color='black', linestyle='--')
axes[0].set_xlabel('positive-detection span [days]')
quality['max_positive_gap_days'].hist(ax=axes[1], bins=50)
axes[1].set_xlabel('largest gap between positive detections [days]')
for axis in axes: axis.set_ylabel('detected sources')
fig.tight_layout()

## Criteria sweep comparison

The sweep output directories encode SNR threshold, minimum detections, and maximum timespan. Rate-limit summaries can be compared against the quality distributions to identify criteria combinations that change the result for physical reasons versus timing or cadence artifacts.

In [ ]:
import re
from pathlib import Path

rows = []
for path in Path('results/detection_criteria_sweep').glob('*/Bu2026Fixed/*.json'):
    match = re.search(r'snr([0-9.]+)_ndet(\d+)_span([0-9.]+)d', str(path))
    if match:
        rows.append({'file': str(path), 'snr_threshold': float(match.group(1)), 'min_detections': int(match.group(2)), 'max_timespan_days': float(match.group(3))})
criteria = pd.DataFrame(rows)
criteria